# ANA-049: RUN-127 hashed GNK benchmark generator knob-map review

Collect and interpret the `RUN-127` hashed GNK forward-generator output tables for `EXP-051`, testing whether retained core/readout guardrails and epistasis-weight knob maps match the scratch-plan WS-B acceptance checks.


## Linked experiments/runs

- Experiments: `EXP-051`
- Runs: `RUN-127`


In [ ]:
from pathlib import Path
import json

try:
    analysis_dir = Path(__file__).resolve().parents[1]
except NameError:
    analysis_dir = Path.cwd().resolve()
    if analysis_dir.name == 'notebooks':
        analysis_dir = analysis_dir.parent
    while not (analysis_dir / 'source_tables').exists() and analysis_dir != analysis_dir.parent:
        analysis_dir = analysis_dir.parent

source_dir = analysis_dir / 'source_tables'
table_dir = analysis_dir / 'tables'

print(f'Analysis directory: {analysis_dir}')
print(f'Source tables: {len(list(source_dir.glob("*")))}')


## Derive tables

`derive_tables.py` regenerates the checksum audit, monotonicity summaries, core/readout guardrails, threshold-epistasis summaries, and final `metrics.json` from the copied `RUN-127` source tables.


In [ ]:
derive_path = analysis_dir / 'notebooks' / 'derive_tables.py'
exec(derive_path.read_text(), {'__name__': '__main__', '__file__': str(derive_path)})


## Main metrics


In [ ]:
metrics = json.loads((table_dir / 'metrics.json').read_text())
for key in [
    'all_direction_checks_passed',
    'n_direction_pass',
    'n_monotonicity_checks',
    'core_beneficial_decline_passed',
    'core_plateau_guardrail_passed',
    'core_transition_epistasis_enrichment_passed',
    'joint_multipeak_navigable_rows_epi_weight_0_6',
    'mean_local_maxima_epi_weight_0_6',
    'mean_accessible_path_fraction_epi_weight_0_6',
    'mean_walsh_high_order_fraction_epi_weight_0_6',
    'mean_sign_epistasis_fraction_epi_weight_0_6',
]:
    print(f'{key}: {metrics[key]}')


## Interpretation

`RUN-127` supports the forward knob-map part of `EXP-051`: all registered monotonicity checks pass and `epi_weight = 0.6` gives a multi-peaked but still navigable regime. The core/readout guardrail preserves beneficial-fraction decline and readout plateaus at `epi_weight = 0`, but fold-transition sign-epistasis enrichment is mixed rather than a clean pass.
